In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

In [ ]:
def resnet_layer(inputs, num_filters=16, kernel_size=3, strides=1, activation='relu', batch_normalization=True):
    conv = layers.Conv2D(num_filters, kernel_size=kernel_size, strides=strides, 
                         padding='same', kernel_initializer='he_normal',
                         kernel_regularizer=regularizers.l2(1e-4))
    x = conv(inputs)
    if batch_normalization:
        x = layers.BatchNormalization()(x)
    if activation is not None:
        x = layers.Activation(activation)(x)
    return x

In [ ]:
def resnet_v1(input_shape, depth):
    if (depth - 2) % 6 != 0:
        raise ValueError('depth should be 6n+2 (e.g. 20, 32, 44)')
    num_filters = 16
    num_res_blocks = int((depth - 2) / 6)

    inputs = layers.Input(shape=input_shape)
    x = resnet_layer(inputs=inputs)

    # Instantiate the stack of residual units
    for stack in range(3):
        for res_block in range(num_res_blocks):
            strides = 1
            if stack > 0 and res_block == 0:  # first layer but not first stack
                strides = 2  # downsample
            y = resnet_layer(inputs=x, num_filters=num_filters, strides=strides)
            y = resnet_layer(inputs=y, num_filters=num_filters, activation=None)
            
            # Adjusting shortcut for dimension mismatch
            if stack > 0 and res_block == 0:
                x = resnet_layer(inputs=x, num_filters=num_filters, kernel_size=1, 
                                 strides=strides, activation=None, batch_normalization=False)
            x = layers.add([x, y]) # THE SKIP CONNECTION
            x = layers.Activation('relu')(x)
        num_filters *= 2

    # Final Classification Layer
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(10, activation='softmax', kernel_initializer='he_normal')(x)

    return models.Model(inputs=inputs, outputs=outputs)

In [ ]:
model = resnet_v1(input_shape=(32, 32, 3), depth=20)
model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), metrics=['accuracy'])

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import LearningRateScheduler

# 1. Load Data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# 2. Data Augmentation (The secret weapon)
datagen = ImageDataGenerator(
    width_shift_range=0.1,  # Randomly shift images horizontally
    height_shift_range=0.1, # Randomly shift images vertically
    horizontal_flip=True    # Randomly flip images (cats look like cats in a mirror)
)
datagen.fit(x_train)

# 3. Learning Rate Scheduler (Drop the rate as we get closer to the end)
def lr_schedule(epoch):
    lr = 1e-3
    if epoch > 120:
        lr *= 0.5e-3
    elif epoch > 80:
        lr *= 1e-3
    return lr

lr_scheduler = LearningRateScheduler(lr_schedule)

# 4. Re-compile and Train! (Notice we are using datagen.flow now)
model.compile(loss='categorical_crossentropy', 
              optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), 
              metrics=['accuracy'])

print("Starting professional-grade training...")
history = model.fit(
    datagen.flow(x_train, y_train, batch_size=128),
    epochs=150, 
    validation_data=(x_test, y_test),
    callbacks=[lr_scheduler]
)

In [ ]:
import numpy as np

# Create the C++ header file
with open("resnet_weights.h", "w") as f:
    f.write("// Auto-generated ResNet-20 Weights for Vitis HLS\n")
    f.write("#pragma once\n")
    f.write("#include <ap_fixed.h>\n\n")
    
    # Define our hardware data type (16-bit total, 6-bit integer)
    f.write("typedef ap_fixed<16, 6> weight_t;\n\n")

    print("Extracting weights...")
    
    # Loop through every layer in the Keras model
    for i, layer in enumerate(model.layers):
        weights = layer.get_weights()
        
        if len(weights) == 0:
            continue
            
        # 1. Handle Batch Normalization (4 arrays)
        if len(weights) == 4:
            names = ['gamma', 'beta', 'mean', 'variance']
            for j in range(4):
                flat_w = weights[j].flatten()
                f.write(f"const weight_t layer_{i}_{names[j]}[{len(flat_w)}] = {{\n    ")
                f.write(", ".join([f"{w:.6f}" for w in flat_w]))
                f.write("\n};\n\n")
                
        # 2. Handle Conv2D and Dense (1 or 2 arrays)
        elif len(weights) in [1, 2]:
            flat_kernel = weights[0].flatten()
            
            # Write the Weights/Kernel array
            f.write(f"const weight_t layer_{i}_weights[{len(flat_kernel)}] = {{\n    ")
            f.write(", ".join([f"{w:.6f}" for w in flat_kernel]))
            f.write("\n};\n\n")
            
            # If the layer also has a bias
            if len(weights) == 2:
                flat_bias = weights[1].flatten()
                f.write(f"const weight_t layer_{i}_bias[{len(flat_bias)}] = {{\n    ")
                f.write(", ".join([f"{b:.6f}" for b in flat_bias]))
                f.write("\n};\n\n")

print("Done! Check the Colab file explorer on the left to download 'resnet_weights.h'")